# Americas TechGuard — Período 8
## Integração LoRa/Meshtastic, JSON e Alertas Ambientais — Blumenau / Vale do Itajaí

**Rosemeri Borges** — UniSENAI/SC, Campus Florianópolis  
**Trilha B** (software-only / simulação), com firmware ESP32 pronto para a Trilha A.

Cadeia completa executada aqui:

`chuva e vazão observadas → ATG-ENV 1.0 (JSON) → validação → risco (escada oficial AlertaBlu) → codec ATG-C1 (23 B) → malha Meshtastic → gateway → MQTT/JSON → alerta no celular`

Roda no Colab, sem GPU e sem chave de API.

## 0. Setup

In [ ]:
!pip -q install "jsonschema>=4.20"
!git clone -q https://github.com/RoseBorges44/Americas-TechGuard-Semana8.git

In [ ]:
%cd /content/Americas-TechGuard-Semana8
import sys
sys.path.insert(0, "src")
!ls

## 1. Testes automatizados

Antes de qualquer resultado, o código precisa passar. 28 testes: schema, codec, risco, LoRa, malha, MQTT.

In [ ]:
!python -m pytest -q

## 2. Coleta dos dados REAIS

Duas APIs abertas, sem chave:

- **Open-Meteo Archive (ERA5-Land)** → chuva horária observada nas 4 coordenadas reais dos nós.
- **Open-Meteo Flood (GloFAS v4, Copernicus EMS)** → vazão diária do rio na régua de Blumenau.

A janela do evento é escolhida **automaticamente** (maior vazão do período consultado) — sem cherry-picking.

In [ ]:
!python scripts/00_fetch_data.py --from 2023-01-01 --to 2025-12-31

### 2.1 Vazão → cota: uma aproximação declarada

O GloFAS entrega **vazão**, não **cota**. Não existe curva-chave oficial pública da estação.
Em vez de inventar uma, mapeio **percentis da climatologia GloFAS (1994–hoje)** sobre a
**escada oficial de cotas do AlertaBlu**:

| percentil | cota | estágio oficial |
|---|---|---|
| p50   | 1,2 m | Normalidade |
| p90   | 3,0 m | limiar de Observação |
| p97   | 4,0 m | limiar de Atenção |
| p99,3 | 6,0 m | limiar de **Alerta** |
| p99,9 | 8,0 m | limiar de **Alerta Máximo** |

**Isso não é uma curva-chave.** É uma aproximação monotônica, declarada aqui, no código e no `metrics.json`.

In [ ]:
import json

rating = json.load(open("data/raw/rating_curve.json"))
for a in rating["anchors"]:
    print(f"  p{a['percentile']:<5} -> {a['discharge_m3s']:9.1f} m3/s  ->  cota {a['stage_m']:.1f} m")
print()
print(rating["warning"])
print(f"climatologia GloFAS: {rating['n_days']} dias")

## 3. Pipeline fim-a-fim

In [ ]:
win = json.load(open("data/raw/event_window.json"))
print("evento selecionado:", win)

In [ ]:
!python scripts/01_run_pipeline.py --start {win['start']} --end {win['end']}

## 4. O payload JSON (ETAPA 2)

In [ ]:
from atg_mesh.schema import EXAMPLE
from atg_mesh.validator import parse_and_validate

print(json.dumps(EXAMPLE, indent=2, ensure_ascii=False))
print()
print("valido?", parse_and_validate(json.dumps(EXAMPLE)).ok)

### 4.1 Tratamento de erro — os payloads que o validador REJEITA

In [ ]:
bad = {
    "campo minimo ausente":      {k: v for k, v in EXAMPLE.items() if k != "sensor_value"},
    "enum de risco invalido":    {**EXAMPLE, "risk_level": "panic"},
    "timestamp nao-ISO":         {**EXAMPLE, "timestamp": "12/07/2026 14:00"},
    "coordenada fora do Vale":   {**EXAMPLE, "latitude": 48.8566, "longitude": 2.3522},
    "nivel implausivel (999 m)": {**EXAMPLE, "sensor_value": 999.0},
    "unidade incoerente":        {**EXAMPLE, "unit": "mm"},
}

for nome, p in bad.items():
    r = parse_and_validate(json.dumps(p))
    print(f"{nome:28s} aceito={r.ok}  -> {r.errors[0][:70]}")

r = parse_and_validate('{"schema": "atg-env/1.0", ')
print(f"{'JSON malformado':28s} aceito={r.ok}  -> {r.errors[0][:70]}")

## 5. O problema central: o JSON legível NÃO passa pelo rádio (ETAPA 3)

`DATA_PAYLOAD_LEN = 233 B` dentro de um pacote LoRa de 256 B. O ATG-ENV canônico tem ~665 B.

In [ ]:
from atg_mesh import codec
from atg_mesh.config import DATA_PAYLOAD_LEN

for k, v in codec.size_report(EXAMPLE).items():
    if k.startswith("_"):
        continue
    flag = "CABE" if v <= DATA_PAYLOAD_LEN else "NAO "
    print(f"  {flag}  {v:>4} B   {k}")

print()
print("ATG-C1-BIN (hex):", codec.to_c1_bin(EXAMPLE).hex())
print("ATG-C1-B64      :", codec.to_c1_b64(EXAMPLE))
print()
print("roundtrip:")
print(json.dumps(codec.from_c1_bin(codec.to_c1_bin(EXAMPLE)), indent=1))

## 6. Camada física LoRa: tempo no ar e duty cycle

In [ ]:
from atg_mesh import lora

n = len(codec.to_c1_bin(EXAMPLE))
print(f"payload de fio: {n} B")
print()
for p in ["SHORT_FAST", "MEDIUM_FAST", "LONG_FAST", "LONG_SLOW"]:
    print(f"  {p:12s} ToA={lora.time_on_air_s(n, p):6.3f} s   taxa={lora.bitrate_bps(p):7.1f} bps")

print()
print("Duty cycle com reporte adaptativo ao risco (LongFast):")
for risco, seg in {"safe": 3600, "attention": 900, "alert": 300, "critical": 60}.items():
    print(f"  {risco:10s} a cada {seg//60:2d} min -> {100*lora.time_on_air_s(n)/seg:.3f} %")

### 6.1 Validação do modelo de propagação

O modelo (log-distância, PL0 = FSPL(1 km) + 30 dB, n = 3,5) **não foi ajustado** aos dados.
Confronto com a única medida de campo da bibliografia obrigatória — arXiv:2605.20379,
enlace de 2,47 km, TX 22 dBm, antenas stock (~2 dBi), RSSI medido **−110 dBm**:

In [ ]:
q = lora.link(2.0, 2.0, 2.47, los=False)
print("modelo:", q)
print()
print(f"medido em campo: -110.0 dBm   |   erro do modelo: {abs(q['rssi_dbm'] + 110):.1f} dB")

## 7. A malha Meshtastic: mesh vs. estrela (o resultado que justifica a escolha)

In [ ]:
import pandas as pd

m = json.load(open("outputs/metrics.json"))
mesh, star = m["mesh"], m["star_baseline_hop_limit_1"]

df = pd.DataFrame({
    "mesh (hop_limit=3)": mesh["pdr_by_node"],
    "estrela (hop_limit=1, tipo LoRaWAN)": star["pdr_by_node"],
})
df.loc["GLOBAL"] = [mesh["pdr_pct"], star["pdr_pct"]]
display(df)

print()
print(f"saltos medios ......... {mesh['mean_hops']}")
print(f"latencia mediana ...... {mesh['median_latency_s']} s")
print(f"custo do flooding ..... {mesh['mean_tx_per_packet']} transmissoes por pacote")

## 8. Os envelopes MQTT/JSON reais do Meshtastic (ETAPA 4)

In [ ]:
from atg_mesh import meshtastic_json as mj

print("UPLINK  ", mj.uplink_topic())
print(json.dumps(mj.uplink_telemetry(EXAMPLE, rssi=-98.4, snr=8.1, hops=1),
                 indent=1, ensure_ascii=False))
print()
print("DOWNLINK", mj.downlink_topic(), " (exige canal chamado 'mqtt' com downlink habilitado)")
print(json.dumps(mj.downlink_sendtext(EXAMPLE["alert_message"]),
                 indent=1, ensure_ascii=False))

## 9. Os alertas que chegariam no celular

In [ ]:
alerts = [json.loads(l) for l in open("outputs/alerts.jsonl")]
print(f"{len(alerts)} alertas emitidos")
print()
for a in alerts[:8]:
    print(f"[{a['from']:>9s} -> {a['to']:<9s}] {a['chars']:3d} chars")
    print(f"   {a['alert_message']}")
    print()

## 10. Figuras de evidência (ETAPA 5)

In [ ]:
!python scripts/02_make_figures.py
!python scripts/03_make_examples.py

In [ ]:
from IPython.display import Image, display

for f in ["fig1_hidrograma_alertas", "fig2_topologia_mesh", "fig3_tamanhos_payload",
          "fig4_metricas_mesh", "fig5_timeline_risco"]:
    display(Image(f"outputs/figures/{f}.png"))

## 11. Conclusão

1. **O JSON legível não cabe no rádio.** 665 B → 23 B com o ATG-C1. Foi a restrição que
   moldou todo o projeto.
2. **O mesh não é enfeite.** O nó de Vila Itoupava (18,4 km) entrega 54,2% em topologia
   estrela e 100% em mesh. É o argumento quantitativo para Meshtastic em vez de LoRaWAN aqui.
3. **A cota sozinha alerta tarde.** Cota + taxa de variação + chuva, com a escada oficial
   da Defesa Civil de Blumenau.

**Limitações:** nenhuma placa foi ligada; a cota é derivada da vazão GloFAS por mapeamento
de percentis (não é curva-chave); ERA5-Land subestima picos convectivos; o Meshtastic não
protege contra replay.

Detalhes, referências e próximos passos: `README.md`.